In [ ]:
import h5py
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from flax import nnx
from tqdm import trange

from wm.vae import VAE

In [ ]:
latent_dim = 32
batch_size = 256

In [ ]:
with ocp.CheckpointManager("../checkpoints/TKTKTKJTKT") as ckp:
    state = ckp.latest()
    graphdef = nnx.eval_shape(VAE(latent_dim, nnx.Rngs(0)))

    model: VAE = nnx.merge(state, graphdef)
    model.eval()

In [ ]:
d = {"a": 1, "b": 2}
d.pop("a")
d

{'b': 2}

In [ ]:
@nnx.jit
def get_latents(batch: jax.Array):
    latent_dict = model.encode(batch)
    return latent_dict.mu, latent_dict.logvar


with h5py.File("../data/world_models.h5", "r") as f:
    obs_dset = f["observations"]
    print(obs_dset.shape)
    n_datapoints, _, _, _ = obs_dset.shape
    mu_dset = f.create_dataset(
        "latents/vae1/mu",
        shape=(n_datapoints, latent_dim // 2),
        dtype="f4",
        chunks=True,
    )

    logvar_dset = f.create_dataset(
        "latents/vae1/logvar",
        shape=(n_datapoints, latent_dim // 2),
        dtype="f4",
        chunks=True,
    )

    for start in trange(0, n_datapoints, batch_size):
        # TODO: handle final batch

        start = 0
        end = start + batch_size

        batch = jnp.array(obs_dset[start:end])
        mu, logvar = get_latents(batch)

        mu_dset[start:end] = mu
        logvar_dset[start:end] = logvar


(10100000, 64, 64, 3)
